# Countermovement Jump (CMJ): Force Plate + IMU End-to-End Analysis

This notebook reproduces the functions of the current `cmj_jump_analysis` Python project in a form that can be run step by step in VS Code.

The workflow covers:

1. Load and profile raw force-plate and IMU files.
2. Clean timestamps and low-pass filter both signals.
3. label the protocol as CMJ and save session metadata.
4. Detect ground-truth jumps from the force plate.
5. Detect jumps independently from IMU free-fall periods.
6. Synchronize the two clocks and match events.
7. Calculate force-derived impulse, takeoff velocity, flight-time velocity, and jump height.
8. Extract one IMU feature row per matched jump.
9. Train and compare regression models using grouped cross-validation.
10. Save model artifacts, predictions, quality tables, plots, and an HTML report.

**Scientific interpretation:** the force plate supplies the target (ground truth), while IMU-derived statistics are the model predictors. This dataset is useful for a session-personalized model, but true unseen-athlete validation requires reliable athlete IDs from at least three independent athletes.


## 0. Setup and paths

The notebook imports the tested functions from the existing project rather than copying their implementation. Outputs are written to a separate `notebook_outputs` folder, so running the notebook does not overwrite the project's current results.

You can change any path below or set the corresponding environment variable before starting VS Code.


In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PROJECT_DIR = Path(os.getenv(
    "CMJ_PROJECT_DIR",
    "/Users/anselbiju/Documents/KivoMotion-Team17/cmj_jump_analysis",
)).expanduser().resolve()
RAW_DATA_DIR = Path(os.getenv(
    "CMJ_RAW_DATA_DIR",
    "/Users/anselbiju/Downloads/KivoMotion-Team17/Phase 2 Data jumps",
)).expanduser().resolve()
SAMPLE_FORCE = Path(os.getenv(
    "CMJ_SAMPLE_FORCE",
    str(RAW_DATA_DIR / "20260730_CMJ_Test1.csv"),
)).expanduser().resolve()
SAMPLE_IMU = Path(os.getenv(
    "CMJ_SAMPLE_IMU",
    str(RAW_DATA_DIR / "20260730_IMU_Test1.csv"),
)).expanduser().resolve()
OUTPUT_ROOT = Path(os.getenv(
    "CMJ_NOTEBOOK_OUTPUT",
    str(Path.cwd() / "notebook_outputs"),
)).expanduser().resolve()

PROCESSED_DIR = OUTPUT_ROOT / "data" / "processed"
REPORTS_DIR = OUTPUT_ROOT / "reports"
MODELS_DIR = OUTPUT_ROOT / "models"
PLOTS_DIR = OUTPUT_ROOT / "plots"
for folder in (PROCESSED_DIR, REPORTS_DIR, MODELS_DIR, PLOTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

assert (PROJECT_DIR / "src" / "cmj_analysis").is_dir(), f"Project not found: {PROJECT_DIR}"
assert SAMPLE_FORCE.is_file(), f"Force file not found: {SAMPLE_FORCE}"
assert SAMPLE_IMU.is_file(), f"IMU file not found: {SAMPLE_IMU}"

sys.path.insert(0, str(PROJECT_DIR / "src"))
from cmj_analysis.events import detect_force_jumps, detect_imu_jumps
from cmj_analysis.features import extract_imu_features
from cmj_analysis.io import (
    estimate_sampling_rate,
    nominal_clock_offset_s,
    read_force_csv,
    read_imu_csv,
)
from cmj_analysis.modeling import compare_feature_sets
from cmj_analysis.reporting import create_report
from cmj_analysis.signals import assess_imu_quality, preprocess_force, preprocess_imu
from cmj_analysis.sync import match_events

display(pd.DataFrame({
    "setting": ["Project", "Raw data", "Sample force", "Sample IMU", "Notebook output"],
    "path": [PROJECT_DIR, RAW_DATA_DIR, SAMPLE_FORCE, SAMPLE_IMU, OUTPUT_ROOT],
}))


## 1. Load and profile the raw data

The force reader handles the ForceDecks metadata preamble. The IMU reader parses comment metadata, converts milliseconds to seconds, sorts packets, and averages duplicate timestamps. The sampling frequency is estimated from the median positive time difference.


In [ ]:
force_rec = read_force_csv(SAMPLE_FORCE)
imu_rec = read_imu_csv(SAMPLE_IMU)
force_fs = estimate_sampling_rate(force_rec.data["Time"])
imu_fs = estimate_sampling_rate(imu_rec.data["time_s"])

profile = pd.DataFrame([
    {
        "sensor": "Force plate",
        "rows_after_cleaning": len(force_rec.data),
        "sampling_rate_hz": force_fs,
        "duration_s": force_rec.data["Time"].iloc[-1] - force_rec.data["Time"].iloc[0],
        "null_cells": int(force_rec.data.isna().sum().sum()),
        "duplicate_times_remaining": int(force_rec.data["Time"].duplicated().sum()),
    },
    {
        "sensor": "IMU",
        "rows_after_cleaning": len(imu_rec.data),
        "sampling_rate_hz": imu_fs,
        "duration_s": imu_rec.data["time_s"].iloc[-1] - imu_rec.data["time_s"].iloc[0],
        "null_cells": int(imu_rec.data.isna().sum().sum()),
        "duplicate_times_remaining": int(imu_rec.data["time_s"].duplicated().sum()),
    },
])
display(profile.round(3))
display(Markdown("**Force metadata**"))
display(pd.Series(force_rec.metadata, name="value").to_frame())
display(Markdown("**IMU metadata**"))
display(pd.Series(imu_rec.metadata, name="value").to_frame())


## 2. Preprocess the signals

Both streams use a fourth-order, 20 Hz Butterworth low-pass filter applied forward and backward (zero phase). The IMU step also calculates acceleration and gyroscope vector magnitudes. Force and IMU rows remain high-frequency samples here; they are **not** expected to equal the eventual number of jumps.


In [ ]:
force = preprocess_force(force_rec.data, force_fs)
imu = preprocess_imu(imu_rec.data, imu_fs)
imu_quality = assess_imu_quality(imu, imu_fs)
force.to_csv(PROCESSED_DIR / "force_preprocessed.csv", index=False)
imu.to_csv(PROCESSED_DIR / "imu_preprocessed.csv", index=False)

print(f"Saved {len(force):,} force samples and {len(imu):,} IMU samples.")
display(pd.Series(imu_quality, name="value").to_frame())
display(force.head(3))
display(imu.head(3))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), constrained_layout=True)
axes[0].plot(force["Time"], force["Both"], alpha=0.35, label="Raw total force")
axes[0].plot(force["Time"], force["force_n"], linewidth=1.1, label="Filtered force")
axes[0].set(title="Force-plate signal", xlabel="Force elapsed time (s)", ylabel="Force (N)")
axes[0].legend()

axes[1].plot(imu["time_s"], imu["acc_mag_g"], alpha=0.35, label="Raw acceleration magnitude")
axes[1].plot(imu["time_s"], imu["acc_mag_g_f"], linewidth=1.1, label="Filtered magnitude")
axes[1].axhline(0.65, color="crimson", linestyle="--", linewidth=1, label="Default free-fall threshold")
axes[1].set(title="IMU acceleration magnitude", xlabel="IMU elapsed time (s)", ylabel="Acceleration (g)")
axes[1].legend()

signal_plot = PLOTS_DIR / "preprocessed_signals.png"
fig.savefig(signal_plot, dpi=160)
plt.show()
print(f"Saved: {signal_plot}")


## 3. Categorize the recording

The current dataset does not contain a reliable jump-type field, so the protocol is inferred from the force filename: a filename containing `CMJ` is labelled `CMJ`; otherwise it remains `UNLABELLED`. For a future mixed protocol, use an explicit recording manifest containing CMJ, squat jump, drop jump, and athlete ID.


In [ ]:
jump_type = "CMJ" if "CMJ" in force_rec.source.name.upper() else "UNLABELLED"
manifest = {
    "session_id": force_rec.source.stem,
    "athlete_id": force_rec.metadata.get("AthleteId", "unknown"),
    "jump_type": jump_type,
    "force_source": str(force_rec.source),
    "imu_source": str(imu_rec.source),
    "force_metadata": force_rec.metadata,
    "imu_metadata": imu_rec.metadata,
}
manifest_path = PROCESSED_DIR / "session_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
display(pd.Series({k: v for k, v in manifest.items() if not k.endswith("metadata")}).to_frame("value"))


## 4. Detect force-plate jumps (ground truth)

Bodyweight is estimated from loaded, non-impact force samples. Flight is a sustained region below `max(20 N, 5% of bodyweight)` for at least 0.20 s, with a loaded plate immediately before and after the region.

For each accepted jump:

- Net impulse: $J = \int_{t_{onset}}^{t_{takeoff}}(F_z-mg)\,dt$
- Takeoff velocity: $v_{to}=J/m$
- Impulse jump height: $h=v_{to}^{2}/(2g)$
- Flight-time velocity: $v_{flight}=gT_{flight}/2$
- Flight-time height: $h=gT_{flight}^{2}/8$


In [ ]:
mass_kg = float(force_rec.metadata["Weight"]) if "Weight" in force_rec.metadata else None
force_events = detect_force_jumps(force, force_fs, mass_kg=mass_kg)
if force_events.empty:
    raise RuntimeError("No force jumps detected. Check force units, threshold, and loaded regions.")
force_events.to_csv(PROCESSED_DIR / "force_events.csv", index=False)

force_event_cols = [
    "jump_id", "movement_onset_s", "takeoff_s", "landing_s", "flight_time_s",
    "mass_kg", "net_impulse_ns", "takeoff_velocity_m_s", "flight_velocity_m_s",
    "jump_height_impulse_m", "jump_height_flight_m", "peak_force_n",
]
print(f"Detected {len(force_events)} force-plate jumps.")
display(force_events[force_event_cols].round(4))


## 5. Detect jumps from the IMU

A candidate flight is a sustained period where filtered acceleration magnitude is below 0.65 g for at least 0.15 s. Candidate regions less than 1.5 s apart are consolidated so one jump cycle contributes one event. The landing is located from the post-flight acceleration peak.


In [ ]:
imu_events = detect_imu_jumps(imu)
if imu_events.empty:
    raise RuntimeError("No IMU jumps detected. Check IMU placement, units, and free-fall threshold.")
imu_events.to_csv(PROCESSED_DIR / "imu_events.csv", index=False)

print(f"Detected {len(imu_events)} IMU jumps.")
display(imu_events.round(4))


## 6. Synchronize and match force/IMU events

The file start timestamps provide a rough clock-offset prior. Event matching then estimates the mapping from IMU elapsed time to force elapsed time. With at least three pairs, an affine fit can estimate small clock-rate drift. A pair is marked `good` when absolute takeoff synchronization error is at most 0.10 s.


In [ ]:
expected_offset_s = nominal_clock_offset_s(force_rec, imu_rec)
matched, sync = match_events(
    force_events,
    imu_events,
    expected_offset_s=expected_offset_s,
)
matched.to_csv(PROCESSED_DIR / "matched_events.csv", index=False)
(PROCESSED_DIR / "sync_parameters.json").write_text(
    json.dumps(sync, indent=2), encoding="utf-8"
)

print(f"Nominal file-clock offset: {expected_offset_s}")
display(pd.Series(sync, name="value").to_frame())
display(matched[[
    "jump_id", "takeoff_s", "imu_takeoff_s", "imu_takeoff_on_force_s",
    "sync_error_s", "sync_quality",
]].round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.plot(matched["jump_id"], matched["takeoff_s"], "o-", label="Force takeoff")
ax.plot(matched["jump_id"], matched["imu_takeoff_on_force_s"], "s--", label="Aligned IMU takeoff")
ax.set(title="Matched event timing", xlabel="Jump", ylabel="Force clock time (s)")
ax.legend()
sync_plot = PLOTS_DIR / "matched_event_timing.png"
fig.savefig(sync_plot, dpi=160)
plt.show()


## 7. Analyze takeoff velocity and extract ML features

The comparison below is force-derived in two independent ways: impulse–momentum and flight time. Large disagreement may indicate movement-onset error, bodyweight/mass error, threshold error, non-rigid landing posture, or signal synchronization problems.

Feature extraction creates **one row per matched jump**, using an IMU window from 1.25 s before IMU takeoff until 0.35 s after IMU landing. For each acceleration/gyroscope axis and magnitude it calculates mean, standard deviation, minimum, maximum, range, and RMS, plus acceleration and angular-rate areas.


In [ ]:
velocity_analysis = matched[[
    "jump_id", "takeoff_velocity_m_s", "flight_velocity_m_s",
    "jump_height_impulse_m", "jump_height_flight_m",
]].copy()
velocity_analysis["velocity_difference_m_s"] = (
    velocity_analysis["takeoff_velocity_m_s"] - velocity_analysis["flight_velocity_m_s"]
)
velocity_analysis.to_csv(PROCESSED_DIR / "takeoff_velocity_analysis.csv", index=False)

features = extract_imu_features(imu, matched)
features["session_id"] = manifest["session_id"]
features["athlete_id"] = manifest["athlete_id"]
features.to_csv(PROCESSED_DIR / "model_features.csv", index=False)

display(velocity_analysis.round(4))
print(f"Feature table: {features.shape[0]} jump rows × {features.shape[1]} columns")
display(features.head())


In [ ]:
x = np.arange(len(velocity_analysis)) + 1
width = 0.36
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
axes[0].bar(x - width / 2, velocity_analysis["takeoff_velocity_m_s"], width, label="Impulse method")
axes[0].bar(x + width / 2, velocity_analysis["flight_velocity_m_s"], width, label="Flight-time method")
axes[0].set(title="Takeoff velocity cross-check", xlabel="Jump", ylabel="Velocity (m/s)", xticks=x)
axes[0].legend()
axes[1].bar(x - width / 2, velocity_analysis["jump_height_impulse_m"] * 100, width, label="Impulse method")
axes[1].bar(x + width / 2, velocity_analysis["jump_height_flight_m"] * 100, width, label="Flight-time method")
axes[1].set(title="Jump height cross-check", xlabel="Jump", ylabel="Height (cm)", xticks=x)
axes[1].legend()
velocity_plot = PLOTS_DIR / "takeoff_velocity_analysis.png"
fig.savefig(velocity_plot, dpi=160)
plt.show()


## 8. Load the current multi-session batch and run quality checks

Each raw recording contains many signal samples but normally only about five physical jumps. Therefore `force_events.csv`, `imu_events.csv`, `matched_events.csv`, and `model_features.csv` correctly have one row per detected/matched jump—not one row per sensor sample.

The combined batch table is used for modelling. Sessions that failed detection or synchronization remain visible in the status table instead of silently disappearing.


In [ ]:
BATCH_DIR = PROJECT_DIR / "batch_results"
batch_features_path = BATCH_DIR / "all_model_features.csv"
batch_matched_path = BATCH_DIR / "all_matched_events.csv"
batch_status_path = BATCH_DIR / "batch_session_status.csv"
assert batch_features_path.is_file(), "Run the optional batch-processing cell below first."

batch_features = pd.read_csv(batch_features_path)
batch_matched = pd.read_csv(batch_matched_path)
batch_status = pd.read_csv(batch_status_path)

status_counts = batch_status["status"].fillna("ignored").value_counts().rename_axis("status").to_frame("sessions")
batch_summary = pd.DataFrame({
    "metric": [
        "Feature rows / matched jumps", "Unique sessions in features",
        "Unique athlete IDs", "Missing feature cells", "Duplicate jump IDs",
    ],
    "value": [
        len(batch_features), batch_features["session_id"].nunique(),
        batch_features["athlete_id"].nunique(), int(batch_features.isna().sum().sum()),
        int(batch_features["jump_id"].duplicated().sum()),
    ],
})
display(status_counts)
display(batch_summary)
display(batch_status[batch_status["status"].eq("excluded")].fillna(""))


## 9. Train regression models to predict normalized impulse

The target column is `target_impulse_ns_kg`, calculated as force-plate net impulse divided by mass. Numerically, $J/m$ equals takeoff velocity in m/s because $J=m\Delta v$.

Three regressors are evaluated for both `all_axes` and `magnitude_only` feature families. Validation is grouped by athlete when at least three reliable athlete IDs exist; otherwise it is grouped by session. `test_number`, `recording_date`, identity fields and orientation quality labels are always excluded from predictors, regardless of their pandas dtype. The winning model/feature-set combination is selected by lowest grouped-CV RMSE.


In [ ]:
modeling_features = batch_features.copy()
corrected_model_path = MODELS_DIR / "best_impulse_model.joblib"
result = compare_feature_sets(
    modeling_features,
    target="target_impulse_ns_kg",
    model_path=corrected_model_path,
    candidate_models_dir=MODELS_DIR,
)

result.leaderboard.to_csv(REPORTS_DIR / "model_leaderboard.csv", index=False)
result.predictions.to_csv(REPORTS_DIR / "out_of_fold_predictions.csv", index=False)

selected_features = result.best_result.feature_names
print(f"Validation method: {result.validation_method}")
print(f"Research-ready for unseen athletes: {result.research_ready}")
print(f"Selected feature set: {result.best_feature_set}")
print(f"Selected predictor count: {len(selected_features)}")
print("Administrative leakage check:", {
    name: name in selected_features for name in ["test_number", "recording_date"]
})
display(result.leaderboard.round(4))


## 10. Evaluate predictions and generate the analytics report

Lower MAE and RMSE are better; R² closer to 1 is better. These are out-of-fold predictions, so each displayed prediction was made by a model that did not train on that validation group. Because the present `AthleteId` field does not reliably distinguish athletes, the result estimates new-session performance—not proven new-athlete performance.


In [ ]:
best_row = result.leaderboard.iloc[0]
best_name = best_row["model"]
best_feature_set = best_row["feature_set"]
best_predictions = result.predictions[
    result.predictions["model"].eq(best_name)
    & result.predictions["feature_set"].eq(best_feature_set)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)
axes[0].scatter(best_predictions["actual"], best_predictions["predicted"], s=28, alpha=0.75)
lo = min(best_predictions[["actual", "predicted"]].min())
hi = max(best_predictions[["actual", "predicted"]].max())
axes[0].plot([lo, hi], [lo, hi], "k--", linewidth=1)
axes[0].set(
    title=f"{best_feature_set} / {best_name}: out-of-fold predictions",
    xlabel="Actual force-plate J/m (N·s/kg)",
    ylabel="Predicted J/m (N·s/kg)",
)
axes[1].hist(best_predictions["residual"], bins=20, color="#6A9F58", edgecolor="white")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set(title="Residual distribution", xlabel="Actual − predicted", ylabel="Count")
evaluation_plot = PLOTS_DIR / "corrected_model_evaluation.png"
fig.savefig(evaluation_plot, dpi=170)
plt.show()

html_path, png_path = create_report(
    batch_matched,
    result.leaderboard,
    result.predictions,
    result.validation_method,
    result.research_ready,
    REPORTS_DIR,
)
print(f"Model: {corrected_model_path}")
print(f"HTML report: {html_path}")
print(f"PNG report: {png_path}")


## 11. Load the saved model and make a prediction

The saved artifact contains the fitted preprocessing/model pipeline and its exact ordered feature list. In production, a new jump must first pass the same IMU preprocessing, event detection, matching (during validation), windowing, and feature extraction steps.


In [ ]:
artifact = joblib.load(corrected_model_path)
example_X = modeling_features.loc[[0], artifact["features"]]
example_prediction = float(artifact["model"].predict(example_X)[0])
example_actual = float(modeling_features.loc[0, artifact["target"]])

display(pd.DataFrame([{
    "jump_id": modeling_features.loc[0, "jump_id"],
    "actual_force_J_per_mass": example_actual,
    "predicted_force_J_per_mass": example_prediction,
    "absolute_error": abs(example_actual - example_prediction),
}]).round(4))


## Optional: rebuild the batch results from all paired raw files

Leave `RUN_FULL_BATCH = False` for normal notebook runs. Change it to `True` when raw files have been added or when orientation/model logic changes. The batch script pairs current Phase 2 force/IMU recordings, records orientation and timestamp quality, compares full-axis versus magnitude-only models, and excludes sessions with fewer than three well-synchronized events.


In [ ]:
RUN_FULL_BATCH = False
if RUN_FULL_BATCH:
    command = [
        sys.executable,
        str(PROJECT_DIR / "scripts" / "run_batch.py"),
        "--data-dir", str(RAW_DATA_DIR),
        "--output-dir", str(OUTPUT_ROOT / "batch_results_rebuilt"),
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
else:
    print("Batch rebuild skipped. Set RUN_FULL_BATCH = True when you need to reprocess raw files.")


## Optional: predict a new IMU-only recording

After batch training, `scripts/predict_imu.py` can detect jumps and predict normalized impulse/takeoff velocity without a force file. The validated current-data default is acceleration magnitude below 0.65 g for at least 0.15 s. Body mass is optional and is only needed to convert N·s/kg into raw N·s.


In [ ]:
RUN_IMU_ONLY_EXAMPLE = False
if RUN_IMU_ONLY_EXAMPLE:
    prediction_output = OUTPUT_ROOT / "imu_only_example"
    command = [
        sys.executable,
        str(PROJECT_DIR / "scripts" / "predict_imu.py"),
        "--imu", str(SAMPLE_IMU),
        "--model", str(corrected_model_path),
        "--output-dir", str(prediction_output),
        "--expected-jumps", "5",
    ]
    subprocess.run(command, check=True)
    display(pd.read_csv(prediction_output / "imu_predictions.csv").round(4))
else:
    print("IMU-only example skipped. Set RUN_IMU_ONLY_EXAMPLE = True after batch training.")


## Final interpretation and recommended next data

This notebook has executed the complete technical workflow and kept raw samples, detected events, matched jumps, features, predictions, and reports as separate layers.

For higher impulse/takeoff-velocity accuracy, collect:

- A stable anonymous athlete ID, body mass, height, age group, sex (when ethically appropriate), and IMU placement/orientation.
- At least three athletes for valid grouped testing; preferably 15–30+ athletes with repeated sessions and 10–20+ clean jumps per athlete.
- A synchronization trigger visible to both devices, or shared timestamps with millisecond precision.
- Consistent IMU sampling rate, units, firmware, attachment tightness, and anatomical location.
- Trial labels for CMJ/SJ/DJ, loaded/unloaded condition, footwear, fatigue state, and invalid/aborted attempts.
- Force-plate calibration checks and clear definitions for movement onset, takeoff, landing, and impulse integration.


This project run is intentionally limited to the current paired Phase 2 dataset.


In [ ]:
output_inventory = sorted(
    str(path.relative_to(OUTPUT_ROOT))
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)
display(pd.DataFrame({"generated_file": output_inventory}))
print(f"Complete. Notebook outputs are in: {OUTPUT_ROOT}")
